In [ ]:
### Установка
"Устанавливается однократно"

In [ ]:
!pip install mediascope-api-lib

In [ ]:
### Обновление
"Можно пропустить"

In [ ]:
!pip install --upgrade mediascope-api-lib

In [ ]:
### Обновление/установка зависимостей
"Можно пропустить"

In [ ]:
# Checks if the package is installed; skips install when present (no version/ref check).
# Run this cell, then restart the kernel if packages/imports changed.

import importlib.util
import subprocess
import sys

REPO_URL = 'https://github.com/manicko/mko_get_mediascope_data.git'
GIT_REF = 'master'  # selected ref (pinned commit SHA, tag, or branch)

# Optional non-default override examples (uncomment to use):
# GIT_REF = "0743c3d21ed5ac6291ab0a96dfab26db0664c05f"  # pinned delivery commit
# GIT_REF = "vX.Y.Z"  # use only if a tag exists on the remote

FORCE_REINSTALL = False  # set True to reinstall even if the package is already installed

package_name = "mko_get_mediascope_data"
install_spec = f"git+{REPO_URL}@{GIT_REF}"
print(f"Target: {install_spec}")

installed = importlib.util.find_spec(package_name) is not None
if FORCE_REINSTALL:
    print("FORCE_REINSTALL=True -> installation required.")
    do_install = True
elif installed:
    print(f"Package '{package_name}' is already installed; skipping installation.")
    print("To update or change ref, set FORCE_REINSTALL=True (or reinstall manually), then restart the kernel.")
    do_install = False
else:
    print(f"Package '{package_name}' is not installed; installation required.")
    do_install = True

if do_install:
    if "get_ipython" in globals():
        get_ipython().run_line_magic("pip", f'install --upgrade "{install_spec}"')
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", install_spec])
    print("Install finished. Restart the kernel before continuing if packages/imports changed.")
else:
    print("No install needed.")


In [ ]:
# Настройка окружения

In [ ]:
import json
import platform
import shutil
import sys
from pathlib import Path

import json5
import platformdirs

import mko_get_mediascope_data
from mko_get_mediascope_data.core.paths import APP_DIR as PACKAGE_APP_DIR, APP_NAME, AppPaths, PathResolver
from mko_get_mediascope_data.core.progress import NullProgressReporter
from mko_get_mediascope_data.core.services import AppService

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Package version:", getattr(mko_get_mediascope_data, "__version__", "unknown"))
print("Package location (sanitized):", Path(mko_get_mediascope_data.__file__).resolve().parent)
print("Bundled app dir:", PACKAGE_APP_DIR)

In [3]:

#Настройка путей и файлов отчетта


In [4]:
REPORT_FILENAME = "nat_tv_brands_theiss_weekly.json5"

# Windows example (comment only):
# import os
# WORKSPACE_ROOT = Path(os.environ["LOCALAPPDATA"]) / "mko_get_mediascope_data" / "mko_get_mediascope_data" / "settings" / "connections" / "mediascope.json"
# WORKSPACE_ROOT = Path(platformdirs.user_config_dir(APP_NAME)) / "settings" / 
# Example override to user config/external path (Windows example, comment only):
# WORKSPACE_ROOT = Path(r"%LOCALAPPDATA%\mko_get_mediascope_data\mko_get_mediascope_data\")

# Portable workspace root (override as needed).
WORKSPACE_ROOT = Path.cwd() 
SETTINGS_WORKSPACE = WORKSPACE_ROOT / "settings"   # settings/config workspace
REPORTS_DIR = SETTINGS_WORKSPACE / "reports"       # reports source path (workspace-local)
EXPORT_DIR = WORKSPACE_ROOT / "output"             # output/export
CREDENTIALS_PATH = SETTINGS_WORKSPACE /  "connections" / "mediascope.json"

REPORT_PATH = REPORTS_DIR / REPORT_FILENAME

# print("WORKSPACE_ROOT:", WORKSPACE_ROOT)
# print("SETTINGS_WORKSPACE:", SETTINGS_WORKSPACE)
# print("REPORTS_DIR:", REPORTS_DIR)
# print("EXPORT_DIR:", EXPORT_DIR)
# print("REPORT_PATH:", REPORT_PATH)
# print("CREDENTIALS_PATH:", CREDENTIALS_PATH)


In [ ]:
# Minimal completeness checks
required = [
    SETTINGS_WORKSPACE / "app_config.json5",
    SETTINGS_WORKSPACE / "log_config.json5",
    SETTINGS_WORKSPACE / "defaults" / "tv" / "crosstab.json5",
    REPORT_PATH,
    CREDENTIALS_PATH,
    
]
missing = [str(p) for p in required if not p.exists()]
print("Missing required paths:", missing if missing else "none")


In [ ]:
app_paths = AppPaths(user_dir=WORKSPACE_ROOT, app_dir=PACKAGE_APP_DIR, app_name=APP_NAME)
resolver = PathResolver(WORKSPACE_ROOT)
service = AppService(app_paths, resolver)

print("Resolved export folder (from service):", service.export_folder)

In [ ]:
try:
    export_paths = await service.run_report(
        Path(REPORT_PATH),
        progress=NullProgressReporter(),
        export_folder=EXPORT_DIR,  # переопределение папки экспорта
    )
    # run_report returns export target paths; individual entries may be skipped (warned).
    print("RUN STATUS: COMPLETED (check warnings/output verification; entries may be skipped).")
    print("Returned export target paths:", [str(p) for p in export_paths])
except Exception as exc:
    export_paths = []
    # Do not print secrets/auth payloads; show exception type and non-sensitive message.
    print("RUN STATUS: FAILED - no success claim.")
    print(f"{type(exc).__name__}: {exc}")

In [ ]:
# Do not claim success from path existence alone; run_report may skip entries.

def list_files(root: Path):
    if not root.exists():
        return []
    return sorted([p for p in root.rglob("*") if p.is_file()])


files = list_files(EXPORT_DIR)
print("EXPORT_DIR:", EXPORT_DIR)
print("File count:", len(files))
for p in files[:200]:
    try:
        size = p.stat().st_size
    except OSError:
        size = "?"
    print(f"- {p.relative_to(EXPORT_DIR)} ({size} bytes)")
if len(files) > 200:
    print(f"... ({len(files) - 200} more files omitted)")
